# minDeepVariant: A From-Scratch Tutorial

**How to build a variant caller using deep learning — in under 500 lines of PyTorch.**

This notebook walks through every concept behind [Google's DeepVariant](https://github.com/google/deepvariant), reimplemented minimally for education. By the end, you'll understand:

1. Why variant calling can be treated as **image classification**
2. How DNA pileups are encoded as **grayscale tensors**
3. How a **CNN** learns to detect SNPs and deletions
4. How genomic coordinates translate to **protein consequences**
5. How a **tiered annotation system** separates known from novel variants

---

### Prerequisites
```bash
pip install mindeepvariant
# OR: pip install torch pysam biopython matplotlib
```

## Part 1: The Core Insight — DNA as Images

### What is a pileup?

When we sequence a genome, we get millions of short reads (100-300bp). These reads are aligned to a reference genome, creating a **pileup** — a stack of reads covering each genomic position.

```
Position:    ...  8   9  [10]  11  12  ...
Reference:   ...  A   C  [G]   T   A   ...    ← the "truth"
Read 1:      ...  A   C  [G]   T   A   ...    ← matches
Read 2:      ...  A   C  [T]   T   A   ...    ← MISMATCH!
Read 3:      ...  A   C  [T]   T   A   ...    ← MISMATCH!
Read 4:      ...  A   C  [G]   T   A   ...    ← matches
Read 5:      ...  A   C  [T]   T   A   ...    ← MISMATCH!
```

At position 10, reads 2, 3, and 5 show `T` instead of the reference `G`. This is a **variant** — specifically a heterozygous SNP (some reads match, some don't).

### The DeepVariant insight

Google realized that if you assign each base a **color** (pixel intensity), this pileup becomes an **image**. Mutations appear as **vertical color streaks** — exactly the kind of pattern CNNs excel at detecting.

Let's see this in action.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Our DNA-to-pixel encoding
# Each base gets a unique grayscale intensity
from mindeepvariant.model import BASE_TO_PIXEL

print("DNA Pixel Encoding:")
for base, value in BASE_TO_PIXEL.items():
    bar = '█' * int(value * 20)
    print(f"  {base} → {value:.2f}  {bar}")

### Why these specific values?

The values are **arbitrary but fixed**. The CNN doesn't care that A=0.25 vs A=0.75 — it only needs each base to have a **distinct, consistent** intensity so mutations create detectable contrast.

The key design choices:
- `N` (unknown) = 0.0 (black — no information)
- `-` (deletion) = 0.10 (near-black but distinguishable from N)
- `A, C, G, T` = evenly spaced across 0.25–1.0

Now let's generate all 4 variant classes and visualize them:

In [ ]:
from mindeepvariant.train import generate_synthetic_pileup
from mindeepvariant.model import CLASS_NAMES

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

for class_idx in range(4):
    tensor = generate_synthetic_pileup(class_idx, window=21, depth=30)
    ax = axes[class_idx]
    ax.imshow(tensor.numpy(), cmap='gray', aspect='auto', vmin=0, vmax=1)
    ax.set_title(f"Class {class_idx}: {CLASS_NAMES[class_idx]}", fontsize=13, fontweight='bold')
    ax.set_ylabel("Row 0=Ref, 1-30=Reads")
    ax.set_xlabel("Genomic Position")
    # Mark the center column where the variant is
    ax.axvline(x=10, color='cyan', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axhline(y=0.5, color='red', linewidth=1.5)

plt.suptitle("The 4 Variant Classes as Pileup Images", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nWhat to look for at the CENTER COLUMN (cyan dashed line):")
print("  Class 0 (Hom-Ref):    Uniform color — all reads match reference")
print("  Class 1 (Het-SNP):    Mixed colors — ~50% reads differ")
print("  Class 2 (Hom-Alt):    Uniform but DIFFERENT from row 0 (reference)")
print("  Class 3 (Deletion):   Dark streak — gap pixels (0.10) dominate")

## Part 2: The CNN Architecture

Now that we can see variants as images, we need a neural network that learns to classify them.

### Why a CNN?

Convolutional Neural Networks are designed to detect **local spatial patterns** — edges, textures, streaks. A mutation in our pileup image is exactly that: a vertical streak of different-colored pixels at the center column.

### Our architecture

We use the simplest CNN that works:

```
Input (1×31×21)
    ↓
Conv2d(1→16, 3×3) + ReLU    ← Detect local patterns (edges/streaks)
    ↓
MaxPool(2×2)                  ← Compress, keep strongest signals
    ↓
Conv2d(16→32, 3×3) + ReLU   ← Combine patterns into higher-level features
    ↓
MaxPool(2×2)                  ← Further compression
    ↓
Flatten
    ↓
Linear(→64) + ReLU + Dropout(0.3)  ← Learn classification boundary
    ↓
Linear(→4)                    ← Output: one score per class
    ↓
Softmax                       ← Convert to probabilities
```

**Key difference from the original notebook:** The FC layer dimensions are computed **dynamically** by passing a dummy tensor through the conv layers. This means you can change `window` and `depth` without manually recalculating.

In [ ]:
from mindeepvariant.model import MinDeepVariantCNN

model = MinDeepVariantCNN(window=21, depth=30)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Architecture:")
print(f"{'='*50}")
print(model)
print(f"{'='*50}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nFor comparison:")
print(f"  Google DeepVariant (Inception-v3): ~23,000,000 parameters")
print(f"  minDeepVariant:                    {total_params:>13,} parameters")
print(f"  Ratio:                             {23000000/total_params:.0f}x smaller")

## Part 3: Training on Synthetic Data

### The chicken-and-egg problem

To train a variant caller, you need labeled variants. But to get labeled variants, you need a variant caller. Google solved this using expensive truth sets (Genome in a Bottle). We solve it with **synthetic data**.

### How synthetic training works

1. Generate a random reference sequence
2. For each class, simulate reads with known patterns:
   - **Hom-Ref:** Reads match reference (with ~2% sequencing noise)
   - **Het-SNP:** 30-70% of reads carry a random alternate base
   - **Hom-Alt:** ~95% of reads carry the alternate
   - **Deletion:** ~90% of reads show gap pixels at center
3. Train the CNN to classify these patterns via backpropagation

### Limitations of synthetic data (honest assessment)

| What we model | What we don't model |
|---------------|--------------------|
| Base substitutions | Base quality variation |
| Simple deletions | Complex structural variants |
| Random sequencing noise | Systematic error profiles (ONT homopolymers) |
| Variable allele frequencies | Strand bias, mapping quality |

This is sufficient for a proof-of-concept but would need real training data for clinical use.

Let's train:

In [ ]:
from mindeepvariant.train import train_model

# Train for 30 epochs with 1000 synthetic samples each
# This takes ~2 minutes on CPU
model, loss_history = train_model(
    window=21,
    depth=30,
    epochs=30,
    samples_per_epoch=1000,
    learning_rate=0.0005,
    output_path="mindv_weights.pth"
)

print(f"\nFinal loss: {loss_history[-1]:.4f}")

In [ ]:
# Visualize the training curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(loss_history)+1), loss_history, 'b-o', markersize=4, linewidth=2)
plt.xlabel('Epoch', fontsize=13)
plt.ylabel('Cross-Entropy Loss', fontsize=13)
plt.title('minDeepVariant Training Curve', fontsize=15, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss dropped from {loss_history[0]:.4f} → {loss_history[-1]:.4f}")
print(f"That's a {((loss_history[0]-loss_history[-1])/loss_history[0])*100:.1f}% reduction.")

## Part 4: Testing the Trained Model

Let's verify the model actually learned to classify variants correctly. We'll generate fresh synthetic pileups (never seen during training) and check accuracy + confidence.

In [ ]:
import random

# Generate a test set
n_test = 200  # 50 per class
correct = 0
results = {i: {"correct": 0, "total": 0, "confidences": []} for i in range(4)}

for true_label in range(4):
    for _ in range(50):
        tensor = generate_synthetic_pileup(true_label)
        input_tensor = tensor.unsqueeze(0).unsqueeze(0)
        
        pred_class, confidence, probs = model.predict_with_confidence(input_tensor)
        
        results[true_label]["total"] += 1
        results[true_label]["confidences"].append(confidence)
        if pred_class == true_label:
            correct += 1
            results[true_label]["correct"] += 1

print(f"Overall Accuracy: {correct}/{n_test} ({100*correct/n_test:.1f}%)")
print(f"\nPer-Class Breakdown:")
print(f"{'Class':<20} {'Accuracy':<15} {'Avg Confidence'}")
print("-" * 50)
for i in range(4):
    acc = results[i]['correct'] / results[i]['total'] * 100
    avg_conf = sum(results[i]['confidences']) / len(results[i]['confidences'])
    print(f"{CLASS_NAMES[i]:<20} {acc:>5.1f}%         {avg_conf:.3f}")

In [ ]:
# Confusion matrix visualization
confusion = torch.zeros(4, 4, dtype=torch.int32)

for true_label in range(4):
    for _ in range(100):
        tensor = generate_synthetic_pileup(true_label)
        input_tensor = tensor.unsqueeze(0).unsqueeze(0)
        pred_class, _, _ = model.predict_with_confidence(input_tensor)
        confusion[true_label][pred_class] += 1

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(confusion.numpy(), cmap='Blues')

# Labels
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('True', fontsize=13)
ax.set_title('Confusion Matrix (Synthetic Test Set)', fontsize=14, fontweight='bold')

# Numbers in cells
for i in range(4):
    for j in range(4):
        val = confusion[i][j].item()
        color = 'white' if val > 50 else 'black'
        ax.text(j, i, str(val), ha='center', va='center', fontsize=16, color=color, fontweight='bold')

plt.colorbar(im, label='Count')
plt.tight_layout()
plt.show()

## Part 5: From DNA Position to Protein Consequence

Knowing "there's a G→T change at position 7589" isn't clinically useful. We need to translate this into **protein language**: "Aspartate at codon 91 is replaced by Tyrosine (p.D91Y), causing Ofloxacin resistance."

### The translation pipeline

```
Genomic Position (7589)           ← from the CNN
        ↓
Gene Offset (7589 - 7318 = 271)  ← distance from gene start
        ↓
Codon Number (271 ÷ 3 = 90)      ← which codon (0-indexed)
Position in Codon (271 % 3 = 1)  ← which base in the triplet
        ↓
Reference Codon: GAC → Asp (D)   ← fetched from FASTA
Mutant Codon:    TAC → Tyr (Y)   ← substitute the alt base
        ↓
Result: p.D91Y (Missense, HIGH impact)
```

### The reverse strand complication

For genes on the minus strand (like *rpoB*), we must **reverse-complement** the codon before translating. The genomic sequence reads 3'→5' for these genes, so the codon as written in the FASTA is backwards.

In [ ]:
# Demonstrate the translation logic with a concrete example
# (This example works without a FASTA file — we'll compute manually)

from Bio.Seq import Seq

# Example: gyrA position 7589 (G→T) in M. leprae
gene_start = 7318   # gyrA CDS start
mutation_pos = 7589  # Where the SNP is
ref_base = 'G'
alt_base = 'T'

# Step 1: Distance from gene start
dist = mutation_pos - gene_start
print(f"Step 1: Distance from gene start = {mutation_pos} - {gene_start} = {dist} bp")

# Step 2: Which codon?
codon_number = dist // 3
pos_in_codon = dist % 3
print(f"Step 2: Codon {codon_number} (0-indexed), position {pos_in_codon} within codon")
print(f"         → Amino acid position: {codon_number + 1} (1-indexed)")

# Step 3: Build ref and alt codons
# (In real code, we fetch from FASTA — here we use the known codon)
ref_codon = 'GAC'  # Known from the M. leprae genome
alt_codon = list(ref_codon)
alt_codon[pos_in_codon] = alt_base
alt_codon = ''.join(alt_codon)
print(f"Step 3: Ref codon = {ref_codon}, Alt codon = {alt_codon}")

# Step 4: Translate
ref_aa = str(Seq(ref_codon).translate())
alt_aa = str(Seq(alt_codon).translate())
print(f"Step 4: {ref_codon} → {ref_aa} (Aspartate)")
print(f"         {alt_codon} → {alt_aa} (Tyrosine)")

# Step 5: Clinical interpretation
aa_change = f"p.{ref_aa}{codon_number+1}{alt_aa}"
mut_type = "Missense" if ref_aa != alt_aa else "Synonymous"
print(f"\nResult: {aa_change} ({mut_type})")
print(f"Clinical: This is the gyrA D91Y mutation → Ofloxacin resistance in M. leprae")

## Part 6: Tiered Annotation — Known vs. Novel

This is where minDeepVariant differs from traditional "lookup" tools.

### The problem with lookup tools

Tools like Mykrobe and NTM-Profiler work like a dictionary: "Is this exact mutation in our database? Yes → Resistant. No → Susceptible."

This fails when:
- A **new resistance mutation** emerges that hasn't been catalogued yet
- The mutation is at a known position but with a **different alternate allele**
- The organism is **understudied** (like M. leprae, which can't be cultured)

### Our two-tier approach

| Tier | Condition | Action |
|------|-----------|--------|
| **Tier 1: KNOWN** | Position + Alt allele match a catalogued mutation | Report as "Confirmed Resistance" |
| **Tier 2: NOVEL** | CNN calls a variant in a drug-target region, but it's NOT in the database | Flag as "High-Interest — Investigate" |

Tier 2 is the unique value of this tool. The CNN provides visual evidence that a real variant exists, even if no database has seen it before.

In [ ]:
import json

# Load the leprae panel and show its structure
with open('../configs/leprae.json', 'r') as f:
    panel = json.load(f)

print(f"Organism: {panel['organism']}")
print(f"Reference: {panel['reference']}")
print(f"\nTargets:")
for gene, info in panel['targets'].items():
    n_known = len(info.get('known_mutations', {}))
    print(f"  {gene}")
    print(f"    Region: {info['start']}-{info['end']} (strand: {info['strand']})")
    print(f"    Drug:   {info['drug']}")
    print(f"    Known mutations: {n_known}")
    for pos, mut in info.get('known_mutations', {}).items():
        if not pos.endswith('_2'):  # Skip aliases for display
            print(f"      Position {pos}: {mut['ref']}→{mut['alt']} = {mut['aa']} ({mut['literature']})")

In [ ]:
# Simulate the tier classification logic
# (This demonstrates the concept without needing a real BAM file)

from mindeepvariant.annotator import annotate_variant
from mindeepvariant.scanner import VariantCall

# Scenario 1: A KNOWN resistance mutation
known_call = VariantCall(
    patient_id="Patient_55A",
    gene="gyrA_DRDR",
    contig="NC_002677.1",
    position=7589,
    ref_base="G",
    alt_base="T",
    ref_depth=2,
    alt_depth=45,
    total_depth=47,
    allele_frequency=0.96,
    predicted_class=2,
    class_name="Hom-Alt-SNP",
    confidence=0.98,
    probabilities=[0.01, 0.00, 0.98, 0.01]
)

# Scenario 2: A NOVEL variant in the same region
novel_call = VariantCall(
    patient_id="Patient_G101",
    gene="gyrA_DRDR",
    contig="NC_002677.1",
    position=7595,          # Not in our known_mutations dict
    ref_base="A",
    alt_base="G",
    ref_depth=18,
    alt_depth=22,
    total_depth=40,
    allele_frequency=0.55,
    predicted_class=1,
    class_name="Het-SNP",
    confidence=0.87,
    probabilities=[0.05, 0.87, 0.06, 0.02]
)

# Note: annotate_variant needs a real FASTA for codon translation,
# so we'll demonstrate the tier logic manually here
targets = panel['targets']

print("SCENARIO 1: Known Mutation")
print(f"  Position: {known_call.position}, Alt: {known_call.alt_base}")
known_muts = targets['gyrA_DRDR']['known_mutations']
pos_key = str(known_call.position)
if pos_key in known_muts and known_muts[pos_key]['alt'] == known_call.alt_base:
    print(f"  ✅ TIER 1 (KNOWN): Matches {known_muts[pos_key]['aa']}")
    print(f"     Drug: {known_muts[pos_key]['drug']}")
    print(f"     Source: {known_muts[pos_key]['literature']}")

print(f"\nSCENARIO 2: Novel Variant")
print(f"  Position: {novel_call.position}, Alt: {novel_call.alt_base}")
pos_key = str(novel_call.position)
if pos_key not in known_muts:
    print(f"  ⚠️  TIER 2 (NOVEL): Not in database — flagged for investigation")
    print(f"     CNN confidence: {novel_call.confidence:.2f}")
    print(f"     Allele frequency: {novel_call.allele_frequency:.2f}")
    print(f"     This variant is in the gyrA DRDR but has never been reported.")
    print(f"     Recommended: Validate with Sanger sequencing or structural prediction.")

## Part 7: Running on Real Data

With the model trained and saved, you can scan real patient BAMs from the command line:

```bash
# Train (if you haven't already)
mindeepvariant train --epochs 30 --output mindv_weights.pth

# Scan your cohort
mindeepvariant scan \
    --bam_dir /path/to/aligned_bams/ \
    --ref /path/to/reference.fna \
    --panel configs/leprae.json \
    --weights mindv_weights.pth \
    --outdir results/
```

### What gets produced

| Output File | Contents |
|------------|----------|
| `PatientX_report.txt` | Per-gene variant table with class, confidence, AA change, tier |
| `PatientX_plots.pdf` | Pileup tensor visualizations for each detected variant |
| `Master_Clinical_Summary.csv` | Cohort-wide CSV aggregating all patients |

### Extending to other organisms

minDeepVariant is **organism-agnostic**. To use it on *M. tuberculosis*, *S. aureus*, or any other bacterium:

1. Create a new JSON panel with your target genes and known mutations
2. Point `--ref` to the appropriate reference FASTA
3. Run the same `scan` command

The CNN doesn't care about the organism — it only looks at pileup images.

## Part 8: What minDeepVariant Teaches You

This project demonstrates several key concepts:

1. **Representation matters.** The same data (aligned reads) can be processed statistically OR visually. DeepVariant showed that the visual approach can match or exceed statistical methods.

2. **Transfer learning from vision to genomics.** CNNs were designed for photos, but the pattern-detection principles transfer directly to biological data.

3. **The tradeoff between synthetic and real data.** Synthetic data lets you bootstrap quickly, but real-world generalization requires real training data.

4. **Annotation is as important as calling.** Detecting a variant is only half the job — translating it into biological meaning (protein change, drug resistance, clinical tier) is what makes it useful.

5. **The value of "unknown unknowns."** Lookup tools can only find what they already know. A model-based approach can flag variants that *look real* even if they've never been catalogued.

---

### Further Reading

- Poplin et al., "A universal SNP and small-indel variant caller using deep neural networks." *Nature Biotechnology*, 2018.
- Karpathy, [minGPT](https://github.com/karpathy/minGPT) — minimal GPT implementation.
- Hayduk, [minAlphaFold2](https://github.com/chrishayduk/minAlphaFold2) — minimal protein structure prediction.
- WHO, "Surveillance of drug resistance in leprosy," 2023.